# 07 — Multi-site UAV Musanga fractional cover × Google Satellite Embeddings

## Question
Can annual Google Satellite Embeddings predict **UAV-derived fractional Musanga cover** across multiple landscapes?

## Design
- UAV binary masks are the reference truth.
- **Yoko UAV year = 2020**; other products = **2024**.
- Compare **T0** (observation year), **T+1**, and **T0+T1** embeddings.
- Start at **10 m**, then test **20 / 30 / 50 m**.
- Primary continuous metrics: RMSE, MAE, R², Spearman ρ, bias.
- Detection diagnostics at observed cover >1%, >5%, >20%, >50%: ROC-AUC, PR-AUC; F1/precision/recall/balanced accuracy use a training-selected operating threshold.
- Preserve zeros; require high valid-UAV support.
- Cache expensive target and embedding products to Drive.
- Sentinel-2 is deliberately deferred until the embedding-only baseline is understood.

The Mos mosaics are kept as separate products, while evaluation code also carries a landscape ID so adjacent/overlapping Mos products can be grouped for strict transfer tests.

## BLOCK 01 — Setup, Drive and configuration

In [ ]:
!pip -q install earthengine-api geemap rasterio pyproj scikit-learn scipy joblib

from pathlib import Path
import re, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from rasterio.warp import reproject, Resampling, transform_bounds
from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
    roc_auc_score, average_precision_score, f1_score, precision_score,
    recall_score, balanced_accuracy_score)
import joblib, ee, geemap

from google.colab import drive
DRIVE_ROOT=Path("/content/drive"); MYDRIVE=DRIVE_ROOT/"MyDrive"
if not MYDRIVE.exists() or not any(MYDRIVE.iterdir()):
    drive.mount(str(DRIVE_ROOT), force_remount=False)
else:
    print("Google Drive already mounted.")

ROOT=MYDRIVE/"Colab Notebooks"/"Musanga"
UAV_DIR=ROOT/"data"/"multisite_uav"
OUT=ROOT/"work"/"multisite_embedding_fractional_cover"
OUT.mkdir(parents=True,exist_ok=True)
EE_PROJECT="tropsedslu"
SUPPORTS=[10,20,30,50]
MIN_UAV_COVERAGE=0.90
RANDOM_SEED=42
FORCE=False

SITE_META={
    "Yoko":{"obs_year":2020,"landscape":"Yoko"},
    "Mos_1_2":{"obs_year":2024,"landscape":"Mos"},
    "Mos_3_4":{"obs_year":2024,"landscape":"Mos"},
    "Mos_5_6":{"obs_year":2024,"landscape":"Mos"},
    "YB_Carlier":{"obs_year":2024,"landscape":"YB_Carlier"},
}
print("UAV folder:",UAV_DIR)
print("Output:",OUT)

## BLOCK 02 — Discover authoritative binary UAV masks

In [ ]:
masks=sorted(UAV_DIR.glob("*_musanga_binary_mask_threshold_*.tif"))
rows=[]
for p in masks:
    m=re.match(r"(.+)_musanga_binary_mask_threshold_([0-9.]+)\.tif$",p.name,re.I)
    if not m: continue
    site=m.group(1); thr=float(m.group(2))
    meta=SITE_META.get(site,{"obs_year":2024,"landscape":site})
    with rasterio.open(p) as src:
        b=transform_bounds(src.crs,"EPSG:4326",*src.bounds,densify_pts=21)
        rows.append(dict(site=site,landscape=meta["landscape"],obs_year=meta["obs_year"],
                         threshold=thr,path=str(p),crs=str(src.crs),res_x=abs(src.transform.a),
                         res_y=abs(src.transform.e),width=src.width,height=src.height,
                         lon_min=b[0],lat_min=b[1],lon_max=b[2],lat_max=b[3]))
inventory=pd.DataFrame(rows).sort_values("site").reset_index(drop=True)
assert len(inventory)==5, f"Expected 5 masks, found {len(inventory)}"
display(inventory[["site","landscape","obs_year","threshold","res_x","res_y"]])
inventory.to_csv(OUT/"uav_inventory.csv",index=False)

## BLOCK 03 — Earth Engine annual embedding download and cache

Download only the native 10 m annual embeddings. Downloads are split into small band chunks and cached on Drive so interrupted runs can resume safely.

In [ ]:
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

BANDS=[f"A{i:02d}" for i in range(64)]
BANDS_PER_CHUNK=8

def embedding_path(site,year,support=10):
    return OUT/"embeddings"/f"{site}_embedding_{year}_10m.tif"

def embedding_chunk_dir(site,year):
    return OUT/"embeddings"/"_chunks"/f"{site}_{year}_10m"

def valid_embedding_file(path,expected_bands=64):
    try:
        with rasterio.open(path) as ds:
            return ds.count==expected_bands and ds.width>0 and ds.height>0
    except Exception:
        return False

def get_embedding(site,mask_path,year,support=10):
    if support!=10:
        raise ValueError("Only native 10 m embeddings are downloaded; coarser supports reuse this cache.")
    out=embedding_path(site,year,10); out.parent.mkdir(parents=True,exist_ok=True)
    if out.exists() and valid_embedding_file(out) and not FORCE:
        print("Reusing",out.name); return out
    with rasterio.open(mask_path) as src:
        left,bottom,right,top=transform_bounds(src.crs,"EPSG:4326",*src.bounds,densify_pts=21)
    roi=ee.Geometry.Rectangle([left,bottom,right,top],geodesic=False)
    col=(ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
         .filterDate(f"{year}-01-01",f"{year+1}-01-01").filterBounds(roi))
    print(site,year,"images:",col.size().getInfo())
    img=col.mosaic().select(BANDS)
    cdir=embedding_chunk_dir(site,year); cdir.mkdir(parents=True,exist_ok=True)
    chunks=[]
    for start in range(0,64,BANDS_PER_CHUNK):
        bands=BANDS[start:start+BANDS_PER_CHUNK]
        cp=cdir/f"bands_{start:02d}_{start+len(bands)-1:02d}.tif"
        if not (cp.exists() and valid_embedding_file(cp,len(bands))) or FORCE:
            print(f"  downloading {bands[0]}–{bands[-1]}",flush=True)
            geemap.ee_export_image(img.select(bands),filename=str(cp),scale=10,region=roi,file_per_band=False)
        chunks.append(cp)
    with rasterio.open(chunks[0]) as first:
        profile=first.profile.copy(); profile.update(count=64,dtype="float32",compress="deflate")
        ref=(first.width,first.height,first.crs,first.transform)
    with rasterio.open(out,"w",**profile) as dst:
        b=1
        for cp in chunks:
            with rasterio.open(cp) as ds:
                if (ds.width,ds.height,ds.crs,ds.transform)!=ref:
                    raise RuntimeError(f"Embedding chunk grid mismatch: {cp}")
                arr=ds.read().astype("float32")
                dst.write(arr,indexes=list(range(b,b+ds.count))); b+=ds.count
    if not valid_embedding_file(out): raise RuntimeError(f"Embedding assembly failed: {out}")
    print("COMPLETE",out.name)
    return out

print("EE embedding cache ready.")

## BLOCK 04 — UAV fractional-cover targets on the embedding grid

Build the UAV fractional Musanga-cover target and UAV valid-coverage raster for each requested support, aligned to the cached 10 m embedding grid.

In [ ]:
TARGET_VERSION="v2"

def target_paths(site,year,support):
    d=OUT/"targets"; d.mkdir(parents=True,exist_ok=True)
    return (d/f"{site}_fraction_{support}m_{TARGET_VERSION}.tif",
            d/f"{site}_validcoverage_{support}m_{TARGET_VERSION}.tif")

def target_grid(emb_path,support):
    with rasterio.open(emb_path) as ref:
        factor=int(round(support/10))
        if factor<1 or abs(factor*10-support)>1e-6: raise ValueError("Support must be 10/20/30/50 m")
        h=int(np.ceil(ref.height/factor)); w=int(np.ceil(ref.width/factor))
        tr=ref.transform * rasterio.Affine.scale(factor,factor)
        return (h,w),ref.crs,tr,ref.profile.copy()

def build_target(mask_path,emb_path,site,year,support):
    frac_path,cov_path=target_paths(site,year,support)
    if frac_path.exists() and cov_path.exists() and not FORCE: return frac_path,cov_path
    shape,dst_crs,dst_transform,profile=target_grid(emb_path,support)
    frac=np.full(shape,np.nan,dtype="float32")
    cov=np.zeros(shape,dtype="float32")
    with rasterio.open(mask_path) as src:
        nd=src.nodata
        reproject(source=rasterio.band(src,1),destination=frac,
                  src_transform=src.transform,src_crs=src.crs,src_nodata=nd,
                  dst_transform=dst_transform,dst_crs=dst_crs,dst_nodata=np.nan,
                  resampling=Resampling.average,init_dest_nodata=True)
        # Stream a native-resolution 1=valid / 0=nodata raster to disk, then area-average it.
        tmp=OUT/"targets"/f"_tmp_valid_{site}.tif"
        vp=src.profile.copy(); vp.update(count=1,dtype="uint8",nodata=None,compress="deflate")
        with rasterio.open(tmp,"w",**vp) as vd:
            for _,win in src.block_windows(1):
                a=src.read(1,window=win)
                valid=np.isfinite(a)
                if nd is not None: valid &= a!=nd
                vd.write(valid.astype("uint8"),1,window=win)
        with rasterio.open(tmp) as vs:
            reproject(source=rasterio.band(vs,1),destination=cov,
                      src_transform=vs.transform,src_crs=vs.crs,
                      dst_transform=dst_transform,dst_crs=dst_crs,
                      resampling=Resampling.average,dst_nodata=0,init_dest_nodata=True)
        tmp.unlink(missing_ok=True)
    good=np.isfinite(frac)&(frac>=0)&(frac<=1)&(cov>=MIN_UAV_COVERAGE)
    prof=profile.copy(); prof.update(height=shape[0],width=shape[1],transform=dst_transform,
                                    count=1,dtype="float32",nodata=-9999.0,compress="deflate")
    with rasterio.open(frac_path,"w",**prof) as ds: ds.write(np.where(good,frac,-9999).astype("float32"),1)
    with rasterio.open(cov_path,"w",**prof) as ds: ds.write(cov.astype("float32"),1)
    return frac_path,cov_path

def read_embedding_at_support(path,support,target_shape):
    with rasterio.open(path) as src:
        if support==10: return src.read().astype("float32")
        return src.read(out_shape=(src.count,*target_shape),resampling=Resampling.average).astype("float32")

print("Target builder v2 ready.")

## BLOCK 04A — Interactive QA: native UAV masks vs 10 m fractional cover

Display all five UAV products together on an Esri World Imagery basemap. Each native binary mask and its derived 10 m fractional-cover target can be toggled independently. This is display-only QA; no targets are recomputed or overwritten.

In [ ]:
import io, base64
import matplotlib.colors as mcolors
import folium
from folium.raster_layers import ImageOverlay
from branca.colormap import LinearColormap
from rasterio.warp import transform_bounds
from rasterio.enums import Resampling

SITES=["Mos_1_2","Mos_3_4","Mos_5_6","YB_Carlier","Yoko"]

def _wgs84_bounds(src):
    left,bottom,right,top=transform_bounds(src.crs,"EPSG:4326",*src.bounds,densify_pts=21)
    return [[bottom,left],[top,right]]

def _png_url(arr,cmap,vmin=0,vmax=1,alpha=None):
    rgba=cmap(mcolors.Normalize(vmin=vmin,vmax=vmax)(arr))
    rgba[...,3]=np.isfinite(arr).astype(float) if alpha is None else alpha
    buf=io.BytesIO()
    plt.imsave(buf,np.clip(rgba*255,0,255).astype(np.uint8),format="png")
    return "data:image/png;base64,"+base64.b64encode(buf.getvalue()).decode("ascii")

native_cmap=mcolors.ListedColormap([(0,0,0,0),(1,0,0,1)])
fraction_cmap=plt.get_cmap("YlOrRd")
layers=[]; south=[]; west=[]; north=[]; east=[]

print("="*80)
print("ALL-PRODUCT UAV / 10 m FRACTION SANITY MAP")
print("="*80)

for site in SITES:
    row=inventory.loc[inventory.site==site].iloc[0]
    mask_path=Path(row.path); year=int(row.obs_year)
    frac_path,_=target_paths(site,year,10)

    with rasterio.open(mask_path) as src:
        nbounds=_wgs84_bounds(src)
        factor=max(1,int(np.ceil(max(src.width,src.height)/3000)))
        native=src.read(1,out_shape=(int(np.ceil(src.height/factor)),int(np.ceil(src.width/factor))),
                        resampling=Resampling.nearest).astype(np.float32)
        if src.nodata is not None: native[native==src.nodata]=np.nan
        nres=src.res

    with rasterio.open(frac_path) as src:
        fbounds=_wgs84_bounds(src)
        fraction=src.read(1).astype(np.float32)
        if src.nodata is not None: fraction[fraction==src.nodata]=np.nan
        fres=src.res; fshape=(src.height,src.width)

    native_png=_png_url(native,native_cmap,0,1,np.where(native==1,0.80,0.0))
    fraction_png=_png_url(fraction,fraction_cmap,0,1,np.where(np.isfinite(fraction),0.70,0.0))

    ok=np.isclose(abs(fres[0]),10,atol=.01) and np.isclose(abs(fres[1]),10,atol=.01)
    print(f"{site:12s} year={year}  UAV={nres}  target={fres}  shape={fshape}  "
          f"mean={np.nanmean(fraction):.4f}  10m_QA={'PASS' if ok else 'WARNING'}")

    layers.append((site,native_png,nbounds,fraction_png,fbounds))
    south += [nbounds[0][0],fbounds[0][0]]; west += [nbounds[0][1],fbounds[0][1]]
    north += [nbounds[1][0],fbounds[1][0]]; east += [nbounds[1][1],fbounds[1][1]]

extent=[min(south),min(west),max(north),max(east)]
centre=[(extent[0]+extent[2])/2,(extent[1]+extent[3])/2]
m=folium.Map(location=centre,tiles=None,control_scale=True)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri World Imagery",name="Esri satellite",overlay=False,control=True,max_zoom=20
).add_to(m)

for site,native_png,nbounds,fraction_png,fbounds in layers:
    g=folium.FeatureGroup(name=f"{site} — native UAV binary",show=False)
    ImageOverlay(native_png,bounds=nbounds,opacity=1.0,interactive=True,cross_origin=False,zindex=4).add_to(g)
    g.add_to(m)

    g=folium.FeatureGroup(name=f"{site} — 10 m fraction",show=True)
    ImageOverlay(fraction_png,bounds=fbounds,opacity=.65,interactive=True,cross_origin=False,zindex=5).add_to(g)
    g.add_to(m)

LinearColormap(["#ffffcc","#fd8d3c","#bd0026"],vmin=0,vmax=1,
               caption="Musanga fractional cover (10 m)").add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
m.fit_bounds([[extent[0],extent[1]],[extent[2],extent[3]]])
display(m)

## BLOCK 05 — Build multi-site modelling table

Combine the UAV fractional-cover target with T0 and T+1 embedding features. The 10 m embeddings are reused from Drive; coarser supports use local aggregation rather than additional Earth Engine downloads.

In [ ]:
TABLE_VERSION="v2"

def build_site_table(row,support=10):
    site=row.site; year=int(row.obs_year); mask=Path(row.path)
    e0=get_embedding(site,mask,year,10)
    e1=get_embedding(site,mask,year+1,10)
    frac_path,cov_path=build_target(mask,e0,site,year,support)
    with rasterio.open(frac_path) as ds:
        y=ds.read(1).astype("float32"); nd=ds.nodata; tr=ds.transform
        valid=np.isfinite(y)&(y!=nd)
    with rasterio.open(cov_path) as ds:
        cov=ds.read(1).astype("float32"); cnd=ds.nodata
        valid &= np.isfinite(cov)&(cov!=cnd)&(cov>=MIN_UAV_COVERAGE)
    X0=read_embedding_at_support(e0,support,y.shape)
    X1=read_embedding_at_support(e1,support,y.shape)
    if X0.shape[1:]!=y.shape or X1.shape[1:]!=y.shape:
        raise RuntimeError(f"Grid shape mismatch for {site}: target={y.shape}, T0={X0.shape}, T1={X1.shape}")
    valid &= np.all(np.isfinite(X0),axis=0)&np.all(np.isfinite(X1),axis=0)
    rr,cc=np.where(valid); xs,ys=rasterio.transform.xy(tr,rr,cc,offset="center")
    base={"site":np.repeat(site,len(rr)),"landscape":np.repeat(row.landscape,len(rr)),
          "obs_year":np.repeat(year,len(rr)),"support_m":np.repeat(support,len(rr)),
          "row":rr,"col":cc,"x":np.asarray(xs),"y":np.asarray(ys),
          "uav_valid_fraction":cov[valid],"musanga_fraction":y[valid]}
    feats={**{f"T0_A{j:02d}":X0[j][valid] for j in range(64)},
           **{f"T1_A{j:02d}":X1[j][valid] for j in range(64)}}
    return pd.DataFrame({**base,**feats})

tables=[]
for row in inventory.itertuples(index=False):
    cache=OUT/"tables"/f"{row.site}_10m_{TABLE_VERSION}.csv"; cache.parent.mkdir(parents=True,exist_ok=True)
    if cache.exists() and not FORCE:
        print("Reusing",cache.name); d=pd.read_csv(cache)
    else:
        print("Building",row.site); d=build_site_table(row,10); d.to_csv(cache,index=False)
    tables.append(d)
df10=pd.concat(tables,ignore_index=True)
print("10 m rows:",len(df10))
display(df10.groupby(["landscape","site"]).musanga_fraction.agg(["count","mean","median","max"]))
df10.to_csv(OUT/f"multisite_10m_T0_T1_{TABLE_VERSION}.csv",index=False)

## BLOCK 06 — Metrics: continuous fractional cover + detection diagnostics

In [ ]:
COVER_THRESHOLDS=[0.01,0.05,0.20,0.50]

def reg_metrics(y,p):
    rho=spearmanr(y,p).statistic if len(y)>=3 else np.nan
    return {"RMSE":mean_squared_error(y,p)**0.5,"MAE":mean_absolute_error(y,p),
            "R2":r2_score(y,p),"Spearman":rho,"bias":float(np.mean(p-y))}

def best_f1_cut(y,p,truth_thr):
    z=(y>truth_thr).astype(int)
    best=(0.5,-1)
    for cut in np.linspace(0,1,201):
        f=f1_score(z,p>cut,zero_division=0)
        if f>best[1]: best=(cut,f)
    return best[0]

def detection_metrics(y,p,truth_thr,pred_cut):
    z=(y>truth_thr).astype(int)
    if len(np.unique(z))<2:
        return {"ROC_AUC":np.nan,"PR_AUC":np.nan,"F1":np.nan,"precision":np.nan,
                "recall":np.nan,"balanced_accuracy":np.nan,"pred_cut":pred_cut}
    zh=(p>pred_cut).astype(int)
    return {"ROC_AUC":roc_auc_score(z,p),"PR_AUC":average_precision_score(z,p),
            "F1":f1_score(z,zh,zero_division=0),"precision":precision_score(z,zh,zero_division=0),
            "recall":recall_score(z,zh,zero_division=0),
            "balanced_accuracy":balanced_accuracy_score(z,zh),"pred_cut":pred_cut}

print("Metrics ready.")

## BLOCK 07 — Fast E01/E02/E03: strict leave-one-landscape-out at 10 m

Fast screening version: spatially balanced training sample, 120-tree RF, progress after every fit, and per-fit prediction/model caches. The complete held-out landscape remains untouched for evaluation.

In [ ]:
import gc
FEATURE_SETS={
    "T0":[f"T0_A{i:02d}" for i in range(64)],
    "T+1":[f"T1_A{i:02d}" for i in range(64)],
    "T0+T1":[f"T0_A{i:02d}" for i in range(64)]+[f"T1_A{i:02d}" for i in range(64)]
}

FAST_N_TREES=120
MAX_TRAIN_PER_LANDSCAPE=30000
BENCH_VERSION="v2"
MODEL_DIR=OUT/f"models_fast_{BENCH_VERSION}"; MODEL_DIR.mkdir(parents=True,exist_ok=True)
PRED_DIR=OUT/f"predictions_fast_{BENCH_VERSION}"; PRED_DIR.mkdir(parents=True,exist_ok=True)

def balanced_train_sample(d, train_mask, max_per_landscape=MAX_TRAIN_PER_LANDSCAPE):
    parts=[]
    for landscape,g in d.loc[train_mask].groupby("landscape",sort=True):
        n=min(len(g),max_per_landscape)
        parts.append(g.sample(n=n,random_state=RANDOM_SEED) if len(g)>n else g)
    return pd.concat(parts,ignore_index=True)

reg_rows=[]; det_rows=[]
jobs=[(fs,hold) for fs in FEATURE_SETS for hold in sorted(df10.landscape.unique())]
print(f"Fast benchmark: {len(jobs)} fits; {FAST_N_TREES} trees; max {MAX_TRAIN_PER_LANDSCAPE:,} training cells/landscape")

for job_i,(fs,hold) in enumerate(jobs,1):
    cols=FEATURE_SETS[fs]
    safe_fs=fs.replace("+","plus")
    pred_path=PRED_DIR/f"10m_{safe_fs}_hold_{hold}.npz"
    model_path=MODEL_DIR/f"10m_{safe_fs}_hold_{hold}.joblib"

    te=df10.landscape==hold
    yte=df10.loc[te,"musanga_fraction"].to_numpy()

    if pred_path.exists() and not FORCE:
        z=np.load(pred_path)
        pte=z["pred"]; train_y=z["train_y"]; train_p=z["train_pred"]
        print(f"[{job_i}/{len(jobs)}] {fs:6s} hold={hold:12s} — cached")
    else:
        train=balanced_train_sample(df10,~te)
        Xtr=train[cols].to_numpy(dtype=np.float32)
        ytr=train["musanga_fraction"].to_numpy(dtype=np.float32)
        Xte=df10.loc[te,cols].to_numpy(dtype=np.float32)

        print(f"[{job_i}/{len(jobs)}] {fs:6s} hold={hold:12s} — fit n_train={len(train):,}, n_test={len(yte):,}",flush=True)
        model=RandomForestRegressor(
            n_estimators=FAST_N_TREES,min_samples_leaf=2,max_features="sqrt",
            n_jobs=-1,random_state=RANDOM_SEED
        )
        model.fit(Xtr,ytr)
        pte=np.clip(model.predict(Xte),0,1)
        train_y=ytr
        train_p=np.clip(model.predict(Xtr),0,1)
        joblib.dump(model,model_path,compress=3)
        np.savez_compressed(pred_path,pred=pte.astype("float32"),
                            train_y=train_y.astype("float32"),train_pred=train_p.astype("float32"))
        del Xtr,Xte,model,train
        gc.collect()

    rm=reg_metrics(yte,pte)
    reg_rows.append({"features":fs,"held_out":hold,"n_test":len(yte),**rm})
    print(f"          R2={rm['R2']:.3f} RMSE={rm['RMSE']:.4f} MAE={rm['MAE']:.4f}",flush=True)

    for thr in COVER_THRESHOLDS:
        cut=best_f1_cut(train_y,train_p,thr)
        det_rows.append({"features":fs,"held_out":hold,"truth_threshold":thr,
                         "n_test":len(yte),**detection_metrics(yte,pte,thr,cut)})

reg10=pd.DataFrame(reg_rows)
det10=pd.DataFrame(det_rows)
reg10.to_csv(OUT/"results_10m_regression.csv",index=False)
det10.to_csv(OUT/"results_10m_detection.csv",index=False)

print("\n10 m regression results")
display(reg10)
print("\nMean by temporal feature set")
display(reg10.groupby("features")[["R2","RMSE","MAE","Spearman","bias"]].mean().sort_values("R2",ascending=False))
print("\nSaved restart-safe models/predictions and result CSVs.")

## BLOCK 07A — Diagnostic: within-landscape vs cross-landscape vs baseline

**QA exclusion:** `Mos_5_6` is retained in the canonical dataset but excluded from this diagnostic because visual aggregation QA showed suspicious spatial structure.\n\nDiagnose whether poor strict transfer reflects lack of embedding signal, cross-landscape calibration/domain shift, or spatial alignment. Uses T0 embeddings, a fast RF, within-landscape 80/20 holdout, strict cross-landscape holdout, and a training-mean baseline. The within-landscape split is diagnostic only, not a spatially independent final validation.

In [ ]:
from sklearn.model_selection import train_test_split

# QA exclusion: Mos_5_6 showed suspicious spatial structure in visual aggregation QA.
# Keep it in the canonical dataset, but exclude it from this diagnostic evaluation.
DIAG_EXCLUDE_SITES=["Mos_5_6"]
diag_df=df10.loc[~df10.site.isin(DIAG_EXCLUDE_SITES)].copy()
print("07A QA exclusions:", DIAG_EXCLUDE_SITES)
print(f"Diagnostic rows: {len(diag_df):,} / {len(df10):,}")

DIAG_FEATURES=[f"T0_A{i:02d}" for i in range(64)]
DIAG_TREES=120
DIAG_MAX_TRAIN=30000
DIAG_DIR=OUT/"diagnostics_v2"; DIAG_DIR.mkdir(parents=True,exist_ok=True)

def diag_fit(train,test,label,landscape):
    if len(train)>DIAG_MAX_TRAIN:
        train=train.sample(DIAG_MAX_TRAIN,random_state=RANDOM_SEED)
    Xtr=train[DIAG_FEATURES].to_numpy(dtype=np.float32); ytr=train.musanga_fraction.to_numpy(dtype=np.float32)
    Xte=test[DIAG_FEATURES].to_numpy(dtype=np.float32); yte=test.musanga_fraction.to_numpy(dtype=np.float32)
    m=RandomForestRegressor(n_estimators=DIAG_TREES,min_samples_leaf=2,max_features="sqrt",n_jobs=-1,random_state=RANDOM_SEED)
    m.fit(Xtr,ytr); pred=np.clip(m.predict(Xte),0,1)
    baseline=np.full(len(yte),float(ytr.mean()),dtype=np.float32)
    rows=[]
    for method,p in [("RF",pred),("train_mean_baseline",baseline)]:
        met=reg_metrics(yte,p)
        rows.append({"experiment":label,"landscape":landscape,"method":method,"n_train":len(train),"n_test":len(test),
                     "obs_mean":float(yte.mean()),"pred_mean":float(p.mean()),**met})
    return rows

diag_rows=[]
for landscape in sorted(diag_df.landscape.unique()):
    d=diag_df[diag_df.landscape==landscape]
    itr,ite=train_test_split(np.arange(len(d)),test_size=0.20,random_state=RANDOM_SEED)
    print(f"Within {landscape}: train={len(itr):,}, test={len(ite):,}",flush=True)
    diag_rows += diag_fit(d.iloc[itr],d.iloc[ite],"within",landscape)

for landscape in sorted(diag_df.landscape.unique()):
    train=diag_df[diag_df.landscape!=landscape]; test=diag_df[diag_df.landscape==landscape]
    parts=[]
    for _,g in train.groupby("landscape"):
        parts.append(g.sample(min(len(g),DIAG_MAX_TRAIN),random_state=RANDOM_SEED) if len(g)>DIAG_MAX_TRAIN else g)
    train=pd.concat(parts,ignore_index=True)
    print(f"Cross -> {landscape}: train={len(train):,}, test={len(test):,}",flush=True)
    diag_rows += diag_fit(train,test,"cross",landscape)

diag=pd.DataFrame(diag_rows)
diag.to_csv(DIAG_DIR/"within_cross_baseline.csv",index=False)
display(diag[["experiment","landscape","method","n_train","n_test","obs_mean","pred_mean","R2","RMSE","MAE","Spearman","bias"]])
print("\nRF comparison")
display(diag[diag.method=="RF"].pivot(index="landscape",columns="experiment",values=["R2","Spearman","bias","obs_mean","pred_mean"]))

## BLOCK 07B — Spatial sanity plots: UAV target vs embedding PCA

Plot the 10 m UAV fractional-cover target beside PC1 of the T0 embedding for each product. Row/column indices are used directly, so a conspicuous spatial displacement or inversion should be visible. PC1 is only a visual summary; weak PC1 similarity alone does not imply weak predictive information.

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

PLOT_DIR=DIAG_DIR/"sanity_maps"; PLOT_DIR.mkdir(parents=True,exist_ok=True)

for site in inventory.site:
    d=df10[df10.site==site].copy()
    if d.empty: continue
    h=int(d.row.max())+1; w=int(d.col.max())+1
    target=np.full((h,w),np.nan,dtype=np.float32)
    target[d.row.to_numpy(int),d.col.to_numpy(int)]=d.musanga_fraction.to_numpy(np.float32)
    sample=d.sample(min(len(d),20000),random_state=RANDOM_SEED)
    pca=PCA(n_components=1,random_state=RANDOM_SEED)
    pca.fit(sample[DIAG_FEATURES].to_numpy(np.float32))
    pc=pca.transform(d[DIAG_FEATURES].to_numpy(np.float32))[:,0]
    pcmap=np.full((h,w),np.nan,dtype=np.float32)
    pcmap[d.row.to_numpy(int),d.col.to_numpy(int)]=pc.astype(np.float32)
    fig,ax=plt.subplots(1,2,figsize=(13,5),constrained_layout=True)
    im0=ax[0].imshow(target,vmin=0,vmax=1); ax[0].set_title(f"{site}: UAV Musanga fraction"); ax[0].set_axis_off()
    fig.colorbar(im0,ax=ax[0],fraction=0.046)
    lo,hi=np.nanpercentile(pcmap,[2,98])
    im1=ax[1].imshow(pcmap,vmin=lo,vmax=hi); ax[1].set_title(f"{site}: T0 embedding PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var.)"); ax[1].set_axis_off()
    fig.colorbar(im1,ax=ax[1],fraction=0.046)
    out=PLOT_DIR/f"{site}_target_vs_T0_PC1.png"; fig.savefig(out,dpi=160,bbox_inches="tight")
    plt.show(); plt.close(fig); print("Saved",out)

## BLOCK 08 — Resolution experiment scaffold: 10 / 20 / 30 / 50 m

Run this only after the 10 m baseline above looks sensible. Each support is rebuilt directly from native UAV masks and annual embeddings; it is not derived by averaging the 10 m target.

In [ ]:
resolution_tables={}
for support in SUPPORTS:
    if support==10:
        resolution_tables[10]=df10
        continue
    pieces=[]
    for row in inventory.itertuples(index=False):
        cache=OUT/"tables"/f"{row.site}_{support}m.csv"; cache.parent.mkdir(parents=True,exist_ok=True)
        if cache.exists() and not FORCE:
            d=pd.read_csv(cache)
        else:
            print("Building",row.site,support,"m")
            d=build_site_table(row,support); d.to_csv(cache,index=False)
        pieces.append(d)
    resolution_tables[support]=pd.concat(pieces,ignore_index=True)
    print(support,"m rows:",len(resolution_tables[support]))
print("Resolution target tables ready.")

## BLOCK 09 — Resolution benchmark using the best temporal configuration

Set `BEST_TEMPORAL` after inspecting BLOCK 07. The default is T0, the observation-year embedding.

In [ ]:
BEST_TEMPORAL="T0"
cols=FEATURE_SETS[BEST_TEMPORAL]
support_rows=[]
for support,d in resolution_tables.items():
    for hold in sorted(d.landscape.unique()):
        tr=d.landscape!=hold; te=d.landscape==hold
        model=RandomForestRegressor(n_estimators=400,min_samples_leaf=2,max_features="sqrt",
                                    n_jobs=-1,random_state=RANDOM_SEED)
        model.fit(d.loc[tr,cols].values,d.loc[tr,"musanga_fraction"].values)
        p=np.clip(model.predict(d.loc[te,cols].values),0,1)
        y=d.loc[te,"musanga_fraction"].values
        support_rows.append({"support_m":support,"features":BEST_TEMPORAL,
                             "held_out":hold,"n_test":len(y),**reg_metrics(y,p)})
support_results=pd.DataFrame(support_rows)
display(support_results)
support_results.to_csv(OUT/"results_resolution_regression.csv",index=False)

## BLOCK 10 — Next stage

Do **not** add Sentinel-2 until the embedding-only baseline has been inspected.

Decision sequence:
1. Is fractional cover predictable at 10 m across held-out landscapes?
2. Is T0, T+1, or T0+T1 most robust?
3. Does 20/30/50 m support materially improve transfer?
4. Then add Sentinel-2-only and Embedding + Sentinel-2 benchmarks using the same targets and validation folds.